# ✈️ Personal Travel & Expense Manager

An agentic application built with the **OpenAI Agents SDK** that helps you:

- **Search flights** with real-time web search powered by `WebSearchTool`
- **Track travel expenses** by category with a running total and budget status
- **Send emails** with expense reports or trip summaries via Resend

In [ ]:
# Install dependencies if running in a fresh environment
# %pip install openai-agents gradio dotenv resend --quiet

In [ ]:
import os
import gradio as gr

from manager import TravelExpenseManager
from dotenv import load_dotenv

load_dotenv(override=True)

# Verify the API key is available
if not os.getenv("OPENAI_API_KEY"):
    raise EnvironmentError("OPENAI_API_KEY is not set.")

In [ ]:
manager = TravelExpenseManager(trip_name="Trip Advisor")

In [ ]:
# Gradio UI

with gr.Blocks(
    title="Personal Travel & Expense Manager",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="sky",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
    ),
    css="""
        .header-box { text-align: center; padding: 1.5rem 0 0.5rem; }
        .header-box h1 { font-size: 2rem; font-weight: 700; margin-bottom: 0.2rem; }
        .header-box p  { color: #64748b; font-size: 0.95rem; }
        footer { display: none !important; }
    """,
) as demo:

    with gr.Row(elem_classes="header-box"):
        gr.HTML("""
            <div class="header-box">
                <h1>✈️ Personal Travel &amp; Expense Manager</h1>
                <p>
                    Search flights &nbsp;·&nbsp; Log expenses &nbsp;·&nbsp; Track your budget
                </p>
            </div>
        """)

    with gr.Row():

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Travel Assistant",
                height=520,
                show_copy_button=True,
                render_markdown=True,
                type="messages",
            )
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="e.g. Find me a cheap flight from New York to London next month…",
                    show_label=False,
                    scale=8,
                    lines=1,
                    max_lines=4,
                    autofocus=True,
                )
                send_btn = gr.Button("Send", variant="primary", scale=1, min_width=80)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1, min_width=80)

        with gr.Column(scale=1, min_width=260):

            gr.Markdown("**✈️ Find a flight**")
            examples_flights = gr.Examples(
                examples=[
                    ["What flights can I get from New York to London next month?"],
                    ["Show me cheap economy flights from Los Angeles to Chicago on 2026-04-20"],
                    ["Any business class seats from San Francisco to Tokyo on 2026-05-10?"],
                    ["Find the cheapest round-trip from Miami to Paris in May 2026"],
                ],
                inputs=msg_input,
                label="",
            )
            gr.Markdown("**💸 Track expenses**")
            examples_expenses = gr.Examples(
                examples=[
                    ["I just paid $280 for a hotel room — add it to my expenses"],
                    ["Log $55 for dinner tonight"],
                    ["I spent $30 on a taxi to the airport"],
                    ["Set my total trip budget to $2,500"],
                    ["How much have I spent so far?"],
                ],
                inputs=msg_input,
                label="",
            )
            gr.Markdown("**📧 Send an email**")
            examples_email = gr.Examples(
                examples=[
                    ["Email my expense report"],
                    ["Email a trip summary"],
                    ["Email my budget status"],
                ],
                inputs=msg_input,
                label="",
            )


    async def chat(message: str, chat_history: list):
        """Stream response into the chatbot."""
        if not message.strip():
            yield "", chat_history
            return

        chat_history = chat_history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": ""},
        ]
        yield "", chat_history

        async for chunk in manager.stream_chat(message):
            chat_history[-1]["content"] += chunk
            yield "", chat_history

    send_btn.click(
        fn=chat,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot],
    )

    msg_input.submit(
        fn=chat,
        inputs=[msg_input, chatbot],
        outputs=[msg_input, chatbot],
    )

    clear_btn.click(
        fn=lambda: ([], ""),
        inputs=[],
        outputs=[chatbot, msg_input],
    )

demo.launch(inbrowser=True)